# Huawei-Inspired Healthcare Machine Learning Lab 2
## Linear Regression from Scratch with Gradient Descent

**Healthcare task:** Use body mass index (BMI) to estimate a continuous diabetes disease-progression score.

**Dataset:** Scikit-learn Diabetes Dataset

- Official documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html
- Dataset description: https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset

This lab follows the Huawei HCIA-AI V4.0 expansion experiment structure: read data, initialise parameters, calculate gradients, update parameters with gradient descent, plot the loss curve, and visualise the fitted regression line.

**Educational use only:** This notebook is not a diagnostic or clinical decision-support tool.
## How to read this notebook
- A line starting with `#` is a comment for you. Python ignores it.
- Run the cells in order from top to bottom.
- This notebook shows how the model learns step by step instead of using a ready-made model only.
- Think of gradient descent as repeatedly adjusting the model until its mistakes become smaller.


## Learning objectives

By the end of this lab, learners should be able to:

1. Explain the meaning of slope and intercept.
2. Calculate predictions using a linear equation.
3. Calculate mean squared error.
4. Calculate gradients for the model parameters.
5. update parameters using gradient descent.
6. plot the loss curve.
7. compare a manually built model with scikit-learn.

## Step 1 — Import the required packages

In [ ]:
# NumPy helps us perform calculations with lists of numbers.
import numpy as np
# Pandas helps us display results in clear tables.
import pandas as pd
# Matplotlib draws charts.
import matplotlib.pyplot as plt

# This loads a small healthcare teaching dataset included with scikit-learn.
from sklearn.datasets import load_diabetes
# This divides the data into training and testing groups.
from sklearn.model_selection import train_test_split
# This ready-made model will be used later for comparison.
from sklearn.linear_model import LinearRegression
# These functions measure prediction error and model performance.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# This fixed number helps us get the same split every time we run the notebook.
RANDOM_STATE = 42
print('Packages imported successfully.')

## Step 2 — Load the healthcare dataset

The scikit-learn diabetes dataset contains 442 samples and 10 baseline variables. The target is a quantitative measure of disease progression one year after baseline.

In [ ]:
# Load the diabetes teaching dataset as a table.
diabetes = load_diabetes(as_frame=True)

# X_full contains the input variables, such as BMI and blood measurements.
X_full = diabetes.data.copy()
# y_full contains the disease-progression score we want to predict.
y_full = diabetes.target.copy()

# Show the variable names and the size of the dataset.
print('Feature names:', list(X_full.columns))
print('Dataset shape:', X_full.shape)
# Display the first five rows.
X_full.head()

## Step 3 — Select one feature for visual learning

We use the dataset's standardised `bmi` feature as the input variable and the disease-progression score as the target.

Using one feature makes it possible to draw the regression line clearly.

In [ ]:
# Keep only the BMI feature for the first part of the lab.
# One feature makes it easy to draw and understand the regression line.
X = X_full[['bmi']].to_numpy(dtype=float)
# Convert the target scores into a vertical column of numbers.
y = y_full.to_numpy(dtype=float).reshape(-1, 1)

# Check the shapes so we know the data are arranged correctly.
print('Input shape:', X.shape)
print('Target shape:', y.shape)
print('First five BMI values:')
print(X[:5])

## Step 4 — Split the data

The training set is used to learn the slope and intercept. The test set is reserved for evaluation.

In [ ]:
# Use 80% of the samples for learning and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Show how many samples are in each group.
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## Step 5 — Add the intercept column

Linear regression uses the equation:

$$\hat{y} = b + wx$$

To calculate both parameters in matrix form, we add a column of ones to represent the intercept.

In [ ]:
# Create a helper function that adds a column of ones.
# This extra column allows the model to learn the intercept, also called b.
def add_intercept_column(X_values):
    # Make one '1' for each sample.
    ones = np.ones((X_values.shape[0], 1))
    # Join the column of ones with the original feature values.
    return np.hstack([ones, X_values])

# Apply the helper function to both training and testing data.
X_train_design = add_intercept_column(X_train)
X_test_design = add_intercept_column(X_test)

print('Design matrix shape:', X_train_design.shape)
print(X_train_design[:5])

## Step 6 — Define the prediction function

In [ ]:
# This function calculates the predicted value.
# It multiplies the input values by the model parameters.
def predict(X_design, theta):
    return X_design @ theta

## Step 7 — Define the loss function

This lab uses mean squared error:

$$MSE = \frac{1}{m}\sum(\hat{y}-y)^2$$

In [ ]:
# This function measures the average squared prediction error.
def mean_squared_error_from_scratch(y_true, y_pred):
    # Subtract the correct answer from the prediction.
    errors = y_pred - y_true
    # Square the errors so negative and positive errors do not cancel each other.
    # Then calculate their average.
    return float(np.mean(errors ** 2))

## Step 8 — Define the gradient calculation

The gradient shows the direction in which the parameters should change to reduce the loss.

In [ ]:
# This function calculates how the model parameters should change.
# The result is called the gradient.
def calculate_gradient(X_design, y_true, theta):
    # Count how many training samples we have.
    number_of_samples = X_design.shape[0]
    # Calculate the current predictions.
    y_pred = predict(X_design, theta)
    # Calculate the current mistakes.
    errors = y_pred - y_true
    # Calculate the direction and size of the parameter update.
    gradient = (2 / number_of_samples) * (X_design.T @ errors)
    return gradient

## Step 9 — Initialise the parameters

In [ ]:
# Start with both model parameters equal to zero.
# theta[0] is the intercept and theta[1] is the slope.
theta = np.zeros((X_train_design.shape[1], 1))

print('Initial parameters:')
print(theta)
print('theta[0] is the intercept and theta[1] is the slope.')

## Step 10 — Build the gradient descent function
**In simple words:** The model makes a prediction, checks how wrong it is, slightly changes its parameters, and repeats this process many times.


In [ ]:
# This function repeats the learning process many times.
def gradient_descent(X_design, y_true, theta, learning_rate=0.1, epochs=5000):
    # Store the error after every learning step so we can draw it later.
    loss_history = []

    # An epoch is one complete learning step using all training samples.
    for epoch in range(epochs):
        # Find how the parameters should change.
        gradient = calculate_gradient(X_design, y_true, theta)
        # Move the parameters a small step in the direction that reduces error.
        theta = theta - learning_rate * gradient

        # Calculate the new predictions and the new error.
        y_pred = predict(X_design, theta)
        loss = mean_squared_error_from_scratch(y_true, y_pred)
        # Save the error for the loss chart.
        loss_history.append(loss)

    # Return the learned parameters and the recorded errors.
    return theta, loss_history

## Step 11 — Train the model from scratch

In [ ]:
# The learning rate controls the size of each adjustment.
learning_rate = 0.1
# The number of epochs tells the model how many times to repeat learning.
epochs = 5000

# Train the model using our own gradient-descent function.
final_theta, loss_history = gradient_descent(
    X_train_design,
    y_train,
    theta,
    learning_rate=learning_rate,
    epochs=epochs
)

# Separate the two learned parameters so they are easy to read.
intercept = final_theta[0, 0]
slope = final_theta[1, 0]

print(f'Learned intercept: {intercept:.4f}')
print(f'Learned slope: {slope:.4f}')

## Step 12 — Plot the loss curve

A decreasing curve indicates that gradient descent is reducing the training error.
**How to read the chart:** A falling line means the model's mistakes are becoming smaller. A flat line means learning has mostly stopped.


In [ ]:
# Draw how the model's error changed during learning.
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Mean squared error')
plt.title('Gradient Descent Loss Curve')
plt.show()

## Step 13 — Visualise the fitted line

In [ ]:
# Create many BMI values across the observed range.
x_line = np.linspace(X_train.min(), X_train.max(), 200).reshape(-1, 1)
# Add the intercept column to these values.
x_line_design = add_intercept_column(x_line)
# Calculate the predicted score for each value.
y_line = predict(x_line_design, final_theta)

# Draw the original training points and the learned regression line.
plt.figure(figsize=(8, 6))
plt.scatter(X_train, y_train, alpha=0.55, label='Training samples')
plt.plot(x_line, y_line, linewidth=2, label='Regression line')
plt.xlabel('Standardised BMI feature')
plt.ylabel('Disease-progression score')
plt.title('Linear Regression from Scratch')
plt.legend()
plt.show()

## Step 14 — Evaluate the model on the test set

In [ ]:
# Use the model to predict scores for the test samples.
y_test_pred = predict(X_test_design, final_theta)

# Calculate three common performance measures.
mae = mean_absolute_error(y_test, y_test_pred)
rmse = mean_squared_error(y_test, y_test_pred) ** 0.5
r2 = r2_score(y_test, y_test_pred)

print(f'MAE: {mae:.3f}')
print(f'RMSE: {rmse:.3f}')
print(f'R-squared: {r2:.3f}')

## Step 15 — Compare with scikit-learn

The manually implemented model should produce results close to scikit-learn's linear regression model.

In [ ]:
# Build the same type of model using scikit-learn's ready-made function.
sklearn_model = LinearRegression()
# Train it on the same training data.
sklearn_model.fit(X_train, y_train.ravel())
# Make predictions for the same test data.
sklearn_predictions = sklearn_model.predict(X_test)

# Compare the intercept and slope learned by both methods.
comparison = pd.DataFrame({
    'Parameter': ['Intercept', 'Slope'],
    'From scratch': [intercept, slope],
    'Scikit-learn': [sklearn_model.intercept_, sklearn_model.coef_[0]]
})

comparison

In [ ]:
# Measure the scikit-learn model using the same metrics.
sklearn_mae = mean_absolute_error(y_test, sklearn_predictions)
sklearn_rmse = mean_squared_error(y_test, sklearn_predictions) ** 0.5
sklearn_r2 = r2_score(y_test, sklearn_predictions)

# Display the two models side by side.
metric_comparison = pd.DataFrame({
    'Model': ['From scratch', 'Scikit-learn'],
    'MAE': [mae, sklearn_mae],
    'RMSE': [rmse, sklearn_rmse],
    'R-squared': [r2, sklearn_r2]
})

metric_comparison

## Step 16 — Predict one fictional example

The diabetes dataset uses transformed and standardised feature values. Therefore, this example must use a value on the dataset's standardised BMI scale rather than a raw BMI measurement.

In [ ]:
# Create one fictional BMI value on the dataset's standardised scale.
fictional_standardised_bmi = np.array([[0.05]])
# Add the intercept column, just as we did for the training data.
fictional_design = add_intercept_column(fictional_standardised_bmi)
# Ask the model to predict the disease-progression score.
fictional_prediction = predict(fictional_design, final_theta)[0, 0]

print(f'Predicted disease-progression score: {fictional_prediction:.2f}')

## Optional extension — Use all 10 features

The same matrix-based gradient descent functions can be used with all dataset features. Because the features are already transformed, no additional standardisation is required for this teaching extension.

In [ ]:
# Convert all 10 input features into a numeric array.
X_all = X_full.to_numpy(dtype=float)
# Convert the target into a vertical numeric column.
y_all = y_full.to_numpy(dtype=float).reshape(-1, 1)

# Split the full-feature data into training and testing parts.
X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Add an intercept column to both parts.
X_all_train_design = add_intercept_column(X_all_train)
X_all_test_design = add_intercept_column(X_all_test)

# Start all model parameters at zero.
theta_all = np.zeros((X_all_train_design.shape[1], 1))
# Train the model using all 10 features.
theta_all, all_loss_history = gradient_descent(
    X_all_train_design,
    y_all_train,
    theta_all,
    learning_rate=0.1,
    epochs=5000
)

# Predict the test scores and print the results.
all_predictions = predict(X_all_test_design, theta_all)

print('All-feature model results')
print(f'MAE: {mean_absolute_error(y_all_test, all_predictions):.3f}')
print(f'RMSE: {mean_squared_error(y_all_test, all_predictions) ** 0.5:.3f}')
print(f'R-squared: {r2_score(y_all_test, all_predictions):.3f}')

## Student exercises

1. Change the learning rate to `0.01`, `0.05`, `0.5`, and `1.0`.
2. Compare how quickly the loss decreases.
3. Change the number of epochs.
4. Replace `bmi` with another single feature.
5. Add an early-stopping rule when the loss changes very little.
6. Calculate mean absolute error from scratch.
7. Explain why a good mathematical fit does not automatically make a model clinically useful.

## Responsible-use notes

- This dataset is for teaching and benchmarking.
- The target is a quantitative research measure, not a direct diagnosis.
- The model must not be used to predict outcomes for real patients.
- One feature alone cannot represent the full clinical complexity of diabetes progression.
- Clinical deployment would require representative data, external validation, fairness assessment, governance, and regulatory review.